In [35]:
 # # Import modules
import pandas as pd
import os

In [36]:
# # define path and export file names
Import_dir = "/Users/danielbhuglah/Downloads/"     # Where the NASA Archive and Encyclopedia(EU) extract file are found
Download_dir = "/Users/danielbhuglah/Downloads/"   # Where to export the processed data 

Nasa_import_file = "PS_2026.02.23_07.24.41.CSV"    # Name of the file from the NASA Archive
EU_import_file = "exoplanet.eu_catalog_23-02-26_16_32_37.CSV" # Name of the file from the Exoplanet Encyclopedia

NASA_output_file = "NASA_data_filtered_V2.xlsx"    # Panda export of NASA data after it has been filtered and each planet sized.
EU_output_file = "Encyclopdedia_data_NASA_matched_filtered.xlsx" 

# Exports for debugging at various key stages of process
Debug_df_NASA_defaults = "Debug_df_NASA_defaults.xlsx" # For debug extract of the default NASA Details for each in scope planet.
Debug_df_EU_data_alt_list = "Debug_df_EU_data_alt_list.xlsx" # For debug export the encyclopeadia with the alternate planet names split

Debug_df_EU_data_exploded = "Debug_df_EU_data_exploded.xlsx" # For debug export the encyclopedia after each line is exploded for alt name
Debug_df_EU_planets_not_in_NASA = "Debug_df_EU_planets_not_in_NASA.xlsx" # For debug list of EU planets that are not matched to NASA
Debug_df_EU_planets_matched_to_NASA = "Debug_df_EU_planets_matched_to_NASA.xlsx" # for debug EU planets matched to NASA file.
Debug_df_EU_planets_matched_parms_adj_prefilter = "Debug_df_EU_planets_matched_parms_adj_prefilter.xlsx" # EU planets matched to NASA, values adjusted before filtering 

In [37]:
# # set filter criteria
Filter_ttv_flag = 1
Filter_EarthRadius_lower = 2.0
Filter_EarthRadius_upper = 8.5
Filter_Vmag = 13.0

In [38]:
# ratio of radii . Earth to Jupiter. Carole Haswell Book. used to convert Jupiter Radii to Earth equivalents
radius_earth_m = 6.37e6
radius_jupiter_m = 7.15e7
ratio_earth_to_jupiter = radius_earth_m/radius_jupiter_m

In [39]:
# # Define planet classification ranges, in terms of Earth Radii
# # Kopparapu, R.K. et al. (2018) ‘Exoplanet Classification and Yield Estimates for Direct Imaging Missions’,
# # The Astrophysical journal, 856(2), p. 122. Available at: https://doi.org/10.3847/1538-4357/aab205 
df_NASA_ranges = pd.DataFrame({"lower":[0,0.5,1.0,1.75,3.5,6.0,14.3],\
                               "upper": [0.5,1.0,1.75,3.5,6.0,14.3,9999],\
                               "classification": ["1.Radius below lower bound","2.Rocky planet","3.Super Earth","4.Sub Neptune","5.Neptune","6.Jupiters","7.Radius above upper bound"] \
                              })
intervals = pd.IntervalIndex.from_arrays(df_NASA_ranges["lower"],df_NASA_ranges["upper"],closed="both")

In [40]:
# # Function to set planet classifications

def classify_exoplanet(radius):
    match = df_NASA_ranges.loc[intervals.contains(radius),"classification"]
    return match.iloc[0] if not match.empty else None

In [41]:
# Define function to clean exoplanet names
def clean_name(s):
    return (
        s.str.lower()
         .str.replace(" ", "", regex=False)
         .str.replace("-", "", regex=False)
         .str.replace("_", "", regex=False)
         .str.replace("[", "", regex=False)
         .str.replace("]", "", regex=False)
         .str.replace("'", "", regex=False)
    )

In [42]:
# # Load raw data files - NASA Archive
# use low_memory = False to avoid warning message about  mixed data types in columns 4 and 5 on NASA file
# NASA files downloaded from NASA Exoplanet Archive. 
#           Planetary Systems table: https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=PS
#           CSV Format, Download all rows, download all columns (13 July 2026 - 288 columns)
#           skip first 292 rows of download file in load as contains column information about the table.
#df_NASA_data = pd.read_csv("/Users/danielbhuglah/Downloads/PS_2026.02.23_07.24.41.CSV", skiprows=292, low_memory = False)
import_NASA_file = os.path.join(Import_dir,Nasa_import_file)
df_NASA_data = pd.read_csv(import_NASA_file, skiprows=292, low_memory = False)

In [43]:
# # Load raw data files - Exoplanet Encyclopedia
# # Exoplanet Encyclopedia download file
# #          Catalogue: https://exoplanet.eu/catalog/
# #          CSV download 
# #          Filter criteria: (    (mass:mjup<13 AND mass:mearth>0.05)
# #                             OR (mass=null AND Not radius = null)
# #                             OR (mass=null AND mass_sini:mjup < 13)
# #                           ) AND Not star_name = null
# #          Number of columns 98 13 July 2026  
import_EU_file = os.path.join(Import_dir,EU_import_file)
df_EU_data = pd.read_csv(import_EU_file)

In [44]:
###################################################
# # NASA Processing
###################################################
# NASA - only select exoplanet data where 
# i) default flag is 1 and ii) the planet has been confirmed. iii) the radius is within defined range 
# iv) TTV flag is 1 v) the Vmag is less than or equal to the defined value
df_NASA_data_filtered = df_NASA_data[(df_NASA_data["default_flag"] == 1) &
                            (df_NASA_data["soltype"] == "Published Confirmed") &
                            (df_NASA_data["ttv_flag"] == Filter_ttv_flag) &
                            (df_NASA_data["pl_rade"] >= Filter_EarthRadius_lower) &
                            (df_NASA_data["pl_rade"] <= Filter_EarthRadius_upper) &
                            (df_NASA_data["sy_vmag"] <= Filter_Vmag)
                            ]


In [45]:
# # Set exoplanet classification. Copy the filtered DF to ensure no issues with updating when you do the exoplant classification. 
df_NASA_data_filtered_V2 = df_NASA_data_filtered.copy()
df_NASA_data_filtered_V2["Exoplanet_class"] = df_NASA_data_filtered_V2["pl_rade"].apply(classify_exoplanet)

In [46]:
# # Download the filtered NASA file
Output_File = os.path.join(Download_dir,NASA_output_file)
df_NASA_data_filtered_V2.to_excel(Output_File,index=False)

In [47]:
###################################################
# # EU Processing
###################################################

# Raw data file loaded earlier in code
# Covert the eu planet radius (in Jupiter eqivalents) into earth radius equivalents.
# used to do classifications and normalise with NASA archives
df_EU_data["radius_earth"]= df_EU_data["radius"]/ ratio_earth_to_jupiter

In [48]:
# use the earth radius equivalents to assign exoplanet classificaitons
df_EU_data["Exoplanet_class"] = df_EU_data["radius_earth"].apply(classify_exoplanet)

In [49]:
# Look to join EU and NASA files. 
# TTV_flag is only held on the NASA file. 
# Also, if a planet is missing radius or v_mag in the EU file, try to pull from the NASA file.

# clean names by removing blanks, dashes and underscores. This should help joining the two files.
df_EU_data["clean_name"] = clean_name(df_EU_data["name"])
df_NASA_data["clean_name"] = clean_name(df_NASA_data["pl_name"])

# select default records only from the NASA file
df_NASA_data_defaults = df_NASA_data[df_NASA_data["default_flag"] == 1].copy()

In [50]:
Output_File = os.path.join(Download_dir,Debug_df_NASA_defaults)
df_NASA_data_defaults.to_excel(Output_File,index=False)

In [51]:
# explode the alternate name list from eu data and create one row per exploded name
# e.g 54 Psc b has alternate names '54 Psc Ab' and 'HD 3651 b'
# alt_list is created and contains a list ['54 Psc Ab','HD 3651 b']
df_EU_data["alt_list"]= df_EU_data["alternate_names"].str.split(",")

# For debug, download the Encyclopedia file with the cleaned alternate names
Output_File = os.path.join(Download_dir,Debug_df_EU_data_alt_list)
df_EU_data.to_excel(Output_File,index=False)

In [52]:
# create one row for each alternate name value
df_EU_data_exploded = df_EU_data.explode("alt_list")
# clean alternate names by removing blanks, dashes and underscores. This should help joining the two (EU and NASA) files.
df_EU_data_exploded["clean_alt"] = clean_name(df_EU_data_exploded["alt_list"])

In [53]:
# For debug, download the Encyclopedia file that has been exploded to one row per alternate names
Output_File = os.path.join(Download_dir,Debug_df_EU_data_exploded )
df_EU_data_exploded.to_excel(Output_File,index=False)

In [54]:
# setup columns to be pulled in from NASA data
NASA_cols = ["clean_name","ttv_flag"]

# merge the unexploded EU data with the NASA data. this uses the original planet names (excluding blanks, underscores etc.)
# use an inner join so df_merged_EU_data only included records where data exists in both the EU and NASA files.
df_merged_EU_data = df_EU_data.merge(df_NASA_data_defaults,on="clean_name",how="inner",suffixes=("","_nasa"))

# merge the exploded EU data, using the planet's alternate names with NASA's clean name. If the alternate name does not exist then the record is dropped
df_merged_EU_data_exploded = df_EU_data_exploded.merge(df_NASA_data_defaults,left_on="clean_alt",right_on="clean_name",how="inner",suffixes=("","_nasa"))

# Combine the matched records from the unexploded and exploded dataframes
df_merged_main = pd.concat([df_merged_EU_data,df_merged_EU_data_exploded],ignore_index=True)

In [55]:
# get a list of planets in the EU file, that have NOT matched to the NASA file.
df_EU_planets_not_matched_to_NASA = df_EU_data.merge(df_merged_main[["name"]],on="name",how="left",indicator=True)

df_EU_planets_not_matched_to_NASA = df_EU_planets_not_matched_to_NASA[df_EU_planets_not_matched_to_NASA["_merge"] == "left_only"].drop(columns="_merge")

In [56]:
# download the list of planets in the EU file, that have NOT matched to the NASA file.
Output_File = os.path.join(Download_dir,Debug_df_EU_planets_not_in_NASA)
df_EU_planets_not_matched_to_NASA.to_excel(Output_File,index=False)

In [57]:
# download the Exploded Encyclopedia file that has been matched to the NASA file
Output_File = os.path.join(Download_dir,Debug_df_EU_planets_matched_to_NASA)
df_merged_main.to_excel(Output_File,index=False)

In [58]:
# if the exoplanet radius in the EU file is not blank put this into a new column else take the radius from the NASA value pl_rade.
# Then reclassify the exoplanet based on its size
# same approach for the Vmag.
df_merged_main["radius_earth"] = df_merged_main["radius_earth"].replace("", pd.NA)
df_merged_main["merged_radius_earth"] = df_merged_main["radius_earth"].fillna(df_merged_main["pl_rade"])
# use the earth radius equivalents to assign exoplanet classificaitons
df_merged_main["merged_Exoplanet_class"] = df_merged_main["merged_radius_earth"].apply(classify_exoplanet)

df_merged_main["mag_v"] = df_merged_main["mag_v"].replace("", pd.NA)
df_merged_main["merged_vmag"] = df_merged_main["mag_v"].fillna(df_merged_main["sy_vmag"])

In [59]:
# download the Exploded Encyclopedia file that has been mateched to the NASA file with parameters adjusted for filtering
Output_File = os.path.join(Download_dir,Debug_df_EU_planets_matched_parms_adj_prefilter)
df_merged_main.to_excel(Output_File,index=False)

In [60]:
# # EU Processing
# EU - only select exoplanet data where 
# i) default flag is 1 and ii) the planet has been confirmed. iii) the radius is within defined range 
# iv) TTV flag is 1 v) the Vmag is less than or equal to the defined value
df_EU_data_merged_filtered = df_merged_main[(df_merged_main["planet_status"] == "Confirmed") &
                            (df_merged_main["ttv_flag"] == Filter_ttv_flag) &
                            (df_merged_main["merged_radius_earth"] >= Filter_EarthRadius_lower) &
                            (df_merged_main["merged_radius_earth"] <= Filter_EarthRadius_upper) &
                            (df_merged_main["merged_vmag"] <= Filter_Vmag)
                            ]

In [61]:
# # Download the merged Exoplanet Encyclopedia file merged with the NASA archive
Output_File = os.path.join(Download_dir,EU_output_file)
df_EU_data_merged_filtered.to_excel(Output_File,index=False)